# Notebook 15 — Geoprivacy in Substance Use Service Planning
## Scenario B: Substance Use / Overdose / Mental Health Services

NB10 introduced three public health scenarios and identified **Scenario B** as
the most privacy-sensitive: substance use and overdose data involve stigmatised
populations whose records are more re-identifiable even under modest spatial
aggregation.

This notebook makes that argument concrete by applying the map encryption
pipeline to a **synthetic overdose incident dataset for Philadelphia, PA
(2022)**, parameterised from the Philadelphia Department of Public Health's
*Unintentional Drug Overdose Fatalities in Philadelphia, 2022* report (CHART
Vol. 8 No. 3, September 2023).

The core demonstration compares two parameter regimes:

| Parameter set | `bin_size_m` | Suppression rule | Context |
|---------------|-------------|-----------------|---------|
| **Standard** (NB06 default) | 250 m | none | Appropriate for cholera/COVID cluster response |
| **Scenario B** (this notebook) | 500 m | suppress tiles with < 5 records | Required for stigmatised small populations |

**Prerequisites:** NB06 (complete pipeline), NB10 (ethical scenarios), NB14
(data augmentation methods).


In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from shapely.geometry import Point
from map_encryption import MapEncryption, SchemeParams

warnings.filterwarnings("ignore")
SEED = 2022
rng  = np.random.default_rng(SEED)

# ── Load Philadelphia ZIP code boundaries ──────────────────────────────────
zips = gpd.read_file("data/phila_zipcodes.geojson")

# Death counts from Philadelphia DPH CHART Vol.8 No.3 (September 2023)
# Source: Unintentional Drug Overdose Fatalities in Philadelphia, 2022
ZIP_DEATHS = {
    "19134": 193,   # Kensington / Harrowgate / Port Richmond
    "19140": 85,    # Hunting Park / Tioga / Nicetown
    "19124": 74,    # Frankford / Juniata Park
    "19139": 60,    # West Philadelphia / Cobbs Creek
    "19133": 54,    # North Philadelphia West / Fairhill
    "19132": 50,    # Strawberry Mansion / Nicetown
}
TARGET_ZIPS = list(ZIP_DEATHS.keys())

# ── Generate random points within each ZIP polygon ─────────────────────────
def points_in_polygon(polygon, n, rng):
    minx, miny, maxx, maxy = polygon.bounds
    pts = []
    while len(pts) < n:
        xs = rng.uniform(minx, maxx, n * 5)
        ys = rng.uniform(miny, maxy, n * 5)
        for x, y in zip(xs, ys):
            if polygon.contains(Point(x, y)):
                pts.append((y, x))          # (lat, lon) in WGS84
                if len(pts) >= n:
                    break
    return pts[:n]

zip_polys = {
    row["CODE"]: row["geometry"]
    for _, row in zips[zips["CODE"].isin(TARGET_ZIPS)].iterrows()
}

# Substance proportions — Philadelphia DPH CHART 2023
# opioid+stimulant: 55%, opioid only: 28%, stimulant only: 15%, other: 2%
SUBSTANCE_LABELS  = ["opioid + stimulant", "opioid only", "stimulant only", "other"]
SUBSTANCE_WEIGHTS = [0.55, 0.28, 0.15, 0.02]

# Age group weights — approximate, consistent with median age 48 (PDPH 2022)
# and national CDC WONDER 2022 drug OD age distribution for 25-64 dominance
AGE_LABELS   = ["15–24", "25–34", "35–44", "45–54", "55–64", "65+"]
AGE_WEIGHTS  = [0.05,    0.20,    0.25,    0.25,    0.18,    0.07]

# Month weights — mild winter/spring peak consistent with opioid seasonality
MONTH_WEIGHTS = [0.10, 0.10, 0.09, 0.09, 0.08, 0.07,
                 0.07, 0.07, 0.08, 0.09, 0.08, 0.08]

rows = []
uid  = 0
for zc, n in ZIP_DEATHS.items():
    coords = points_in_polygon(zip_polys[zc], n, rng)
    subst  = rng.choice(SUBSTANCE_LABELS,  size=n, p=SUBSTANCE_WEIGHTS)
    age    = rng.choice(AGE_LABELS,         size=n, p=AGE_WEIGHTS)
    month  = rng.choice(range(1, 13),       size=n, p=MONTH_WEIGHTS)
    for i, (lat, lon) in enumerate(coords):
        rows.append({
            "individual_id": uid,
            "zip_code":      zc,
            "LAT":           round(lat, 6),
            "LON":           round(lon, 6),
            "substance":     subst[i],
            "age_group":     age[i],
            "month":         int(month[i]),
            "year":          2022,
        })
        uid += 1

df = pd.DataFrame(rows)
print(f"Synthetic dataset: {len(df)} records across {df['zip_code'].nunique()} ZIP codes")
print(df["zip_code"].value_counts().sort_index().to_string())


Synthetic dataset: 516 records across 6 ZIP codes
zip_code
19124     74
19132     50
19133     54
19134    193
19139     60
19140     85


---
## Part 1 — Synthetic Dataset and Geographic Context

The synthetic dataset mirrors the geographic distribution of the 516 fatal
overdose deaths recorded across six Philadelphia ZIP codes in 2022 (the six
with the highest counts in the city, representing 37 % of all 1,413 deaths).
Substance involvement proportions follow the PDPH CHART report: 55 % of
records involve both an opioid and a stimulant; fentanyl was present in 96 %
of opioid-involved deaths.

Each record's (LAT, LON) is a random point drawn from a **uniform distribution
within the corresponding ZIP code polygon**, not a real address. This preserves
the geographic shape and density of the real data while generating no
individually attributable coordinates.


In [2]:
# Figure 15a — Synthetic overdose incidents by ZIP code
ZIP_COLOURS = {
    "19134": "#d94801",   # Kensington — darkest (highest count)
    "19140": "#f16913",
    "19124": "#fd8d3c",
    "19133": "#fdae6b",
    "19132": "#fdd0a2",
    "19139": "#feedde",
}

centre = [40.000, -75.130]
m15a = folium.Map(location=centre, zoom_start=13,
                  tiles="cartodbpositron", control_scale=True)

# ZIP code outlines
zip_layer = gpd.read_file("data/phila_zipcodes.geojson")
zip_layer = zip_layer[zip_layer["CODE"].isin(TARGET_ZIPS)]
folium.GeoJson(
    zip_layer.__geo_interface__,
    name="ZIP boundaries",
    style_function=lambda f: {
        "fillColor": ZIP_COLOURS.get(f["properties"]["CODE"], "#cccccc"),
        "color": "#333333", "weight": 1.5, "fillOpacity": 0.15,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["CODE"], aliases=["ZIP code"]
    ),
).add_to(m15a)

# Individual incidents coloured by substance
SUBST_COLOURS = {
    "opioid + stimulant": "#7b2d8b",
    "opioid only":        "#2171b5",
    "stimulant only":     "#d94801",
    "other":              "#888888",
}
subst_fg = {s: folium.FeatureGroup(name=f"Substance: {s}", show=True)
            for s in SUBSTANCE_LABELS}
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row["LAT"], row["LON"]],
        radius=3,
        color=SUBST_COLOURS[row["substance"]],
        fill=True, fill_color=SUBST_COLOURS[row["substance"]],
        fill_opacity=0.7,
        popup=f"ZIP {row['zip_code']} | {row['substance']} | age {row['age_group']}",
    ).add_to(subst_fg[row["substance"]])

for fg in subst_fg.values():
    fg.add_to(m15a)

plugins.Fullscreen(position="topleft").add_to(m15a)
folium.LayerControl(collapsed=False).add_to(m15a)
m15a


**Figure 15a.** Synthetic overdose incident locations for six Philadelphia ZIP
codes in 2022, coloured by substance involvement: purple = opioid + stimulant
(55 %), blue = opioid only (28 %), orange = stimulant only (15 %), grey =
other (2 %). ZIP code boundaries are overlaid with fill intensity proportional
to death count; 19134 (Kensington) is darkest with 193 records. Points are
random within each ZIP polygon, not real addresses.


---
## Part 2 — Standard 250 m Pipeline: Why It Falls Short

The default `SchemeParams(bin_size_m=250)` was designed for the cholera
dataset — 250 deaths clustered in a ~300 m × 300 m area of Soho. Applied to
a stigmatised population spread across 6 km² of North Philadelphia, many
250 m tiles will contain only a single record. A record that occupies a tile
alone has **k = 1** under the k-anonymity model: its tile membership alone
is sufficient to identify it, regardless of PRP shuffle and jitter.

**A note on encrypted tile indices.** Each record is encoded with a unique
per-record tweak (`individual_id`), so the PRP maps the same geographic tile
to *different* encrypted indices (qxp, qyp) for each record — an intentional
privacy property that prevents an adversary who observes the encrypted index
from inferring which records share a geographic bin. k-anonymity is therefore
measured on the **original geographic tile assignment** (which 250 m cell the
record's coordinates fall in), not on the encrypted output.

Below we encode all 516 records at 250 m resolution and compute the
k-anonymity distribution across geographic tiles.

In [3]:
# Encode at 250 m (standard) and compute k-anonymity on original tile indices
#
# Note on tweaks: each record uses individual_id as the PRP tweak, which
# maps every record to a *different* (qxp, qyp) even when two records share
# the same geographic tile. This is a deliberate privacy property — the
# encrypted tile indices do not reveal which records are co-located.
# k-anonymity is therefore computed on the original (qx, qy) tile indices
# (i.e. which geographic 250 m cell each record falls in), not on the
# encrypted indices.

import math
_R = 6378137.0

def _tile(lat, lon, bin_m):
    x = math.radians(lon) * _R
    y = math.log(math.tan(math.pi/4 + math.radians(lat)/2)) * _R
    return int(round(x / bin_m)), int(round(y / bin_m))

KEY_250 = os.urandom(32)
me_250  = MapEncryption(KEY_250, SchemeParams(bin_size_m=250))

encoded_250 = []
for _, row in df.iterrows():
    qx, qy = _tile(row["LAT"], row["LON"], 250)
    rec = me_250.encode(row["LAT"], row["LON"],
                        tweak=MapEncryption.make_tweak(int(row["individual_id"])))
    encoded_250.append({
        "individual_id": row["individual_id"],
        "qx": qx, "qy": qy,           # original tile — used for k-anonymity
        "qxp": rec["qxp"], "qyp": rec["qyp"],
        "nonce": rec["nonce"], "ct_resid": rec["ct_resid"],
        "tweak": rec["tweak"],
    })

enc_250 = pd.DataFrame(encoded_250)

# k-anonymity on original pre-PRP tiles
tile_counts_250 = enc_250.groupby(["qx","qy"]).size().reset_index(name="k")
enc_250 = enc_250.merge(tile_counts_250, on=["qx","qy"])

n_tiles_250    = len(tile_counts_250)
k1_pct_250     = (tile_counts_250["k"] == 1).sum() / n_tiles_250 * 100
k5plus_pct_250 = (tile_counts_250["k"] >= 5).sum() / n_tiles_250 * 100

print(f"250 m — unique geographic tiles : {n_tiles_250}")
print(f"250 m — tiles k=1              : {(tile_counts_250['k']==1).sum()}  ({k1_pct_250:.1f}%)")
print(f"250 m — tiles k>=5             : {(tile_counts_250['k']>=5).sum()}  ({k5plus_pct_250:.1f}%)")
print(f"250 m — records in k=1 tiles   : {(enc_250['k']==1).sum()}")

250 m — unique geographic tiles : 416
250 m — tiles k=1              : 330  (79.3%)
250 m — tiles k>=5             : 0  (0.0%)
250 m — records in k=1 tiles   : 330


In [4]:
# Figure 15b — Original positions coloured by k-anonymity at 250 m
# render_coordinates() returns PRP-shuffled positions scattered globally —
# correct privacy behaviour but not useful for a service-area map.
# Here we colour the *original* incident locations by tile k-value to show
# where re-identifiable singletons (k=1) actually sit geographically.
enc_250_geo = enc_250.merge(df[["individual_id", "LAT", "LON"]], on="individual_id")

m15b = folium.Map(location=centre, zoom_start=13,
                  tiles="cartodbpositron", control_scale=True)

for _, row in enc_250_geo.iterrows():
    colour = "#d94801" if row["k"] == 1 else "#2171b5"
    folium.CircleMarker(
        location=[row["LAT"], row["LON"]],
        radius=3 if row["k"] == 1 else 2,
        color=colour, fill=True,
        fill_color=colour, fill_opacity=0.7,
        popup=f"k={row['k']} (tile count)",
    ).add_to(m15b)

legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;background:white;'
    'padding:8px 12px;border-radius:6px;font-size:12px;'
    'border:1px solid #aaa;z-index:9999;">'
    '<b>k-anonymity (250 m)</b><br>'
    '<span style="color:#d94801;">&#9679;</span> k = 1 (unique tile — re-identifiable)<br>'
    '<span style="color:#2171b5;">&#9679;</span> k ≥ 2 (shared tile)'
    '</div>'
)
m15b.get_root().html.add_child(folium.Element(legend_html))
plugins.Fullscreen(position="topleft").add_to(m15b)
m15b

**Figure 15b.** Original incident locations coloured by tile k-anonymity at 250 m resolution.
Red circles are records whose encrypted tile (qxp, qyp) is unique in the dataset (k = 1):
an adversary who can map encrypted tile indices to geographic regions — for example by
submitting test records to the encode API — can locate these individuals precisely by tile
membership alone, without ever decrypting ct_resid. Blue circles share a tile with at
least one other record. At 250 m resolution the majority of tiles are singletons —
inadequate privacy for a stigmatised population.

In [5]:
# Figure 15c — k-anonymity tile-count distribution at 250 m
k_vals_250 = tile_counts_250["k"].clip(upper=10)
k_hist_250 = k_vals_250.value_counts().sort_index()

fig_15c = go.Figure(go.Bar(
    x=[str(k) if k < 10 else "10+" for k in k_hist_250.index],
    y=k_hist_250.values,
    marker_color=["#d94801" if k == 1 else "#2171b5" for k in k_hist_250.index],
    text=k_hist_250.values, textposition="outside",
))
fig_15c.update_layout(
    title="k-anonymity tile distribution — 250 m bins",
    xaxis_title="Records per tile (k)",
    yaxis_title="Number of tiles",
    template="plotly_white", height=380, width=600,
)
fig_15c.show()


**Figure 15c.** k-anonymity distribution at 250 m: each bar shows how many
encrypted tiles contain exactly k records. Tiles with k = 1 (red) are
completely re-identifiable by tile membership alone — an adversary who can
map encrypted tile indices to geographic regions (e.g., by querying the
scheme with test records) can uniquely locate those individuals. For a
stigmatised population such as people who use drugs, k = 1 is an
unacceptable disclosure risk.


---
## Part 3 — Scenario B Parameters: 500 m Bins and k < 5 Suppression

The NB10 Scenario B recommendation is:

> *Use with caution. Small, stigmatised populations are more re-identifiable
> even at 250 m. Consider `bin_size_m ≥ 500 m` and suppressing cells with
> < 5 records.*

Two changes are made:

1. **`bin_size_m = 500`**: Each tile covers a 500 m × 500 m area (4× the area
   of 250 m tiles), so more records are aggregated per tile.
2. **Suppression rule**: Tiles with fewer than 5 records are suppressed from
   the public display map. Suppressed records exist in the decode tier (for
   licensed analysts) but are not visualised publicly.


In [6]:
# Encode at 500 m (Scenario B) and apply k<5 suppression
KEY_500 = os.urandom(32)
me_500  = MapEncryption(KEY_500, SchemeParams(bin_size_m=500))

encoded_500 = []
for _, row in df.iterrows():
    qx, qy = _tile(row["LAT"], row["LON"], 500)
    rec = me_500.encode(row["LAT"], row["LON"],
                        tweak=MapEncryption.make_tweak(int(row["individual_id"])))
    encoded_500.append({
        "individual_id": row["individual_id"],
        "qx": qx, "qy": qy,
        "qxp": rec["qxp"], "qyp": rec["qyp"],
        "nonce": rec["nonce"], "ct_resid": rec["ct_resid"],
        "tweak": rec["tweak"],
    })

enc_500 = pd.DataFrame(encoded_500)

tile_counts_500 = enc_500.groupby(["qx","qy"]).size().reset_index(name="k")
enc_500 = enc_500.merge(tile_counts_500, on=["qx","qy"])

K_MIN = 5
enc_500["suppressed"] = enc_500["k"] < K_MIN

n_tiles_500    = len(tile_counts_500)
k1_pct_500     = (tile_counts_500["k"] == 1).sum() / n_tiles_500 * 100
k5plus_pct_500 = (tile_counts_500["k"] >= K_MIN).sum() / n_tiles_500 * 100
suppressed_pct = enc_500["suppressed"].mean() * 100

print(f"500 m — unique geographic tiles : {n_tiles_500}")
print(f"500 m — tiles k=1              : {(tile_counts_500['k']==1).sum()}  ({k1_pct_500:.1f}%)")
print(f"500 m — tiles k>=5             : {(tile_counts_500['k']>=5).sum()}  ({k5plus_pct_500:.1f}%)")
print(f"500 m — records suppressed     : {enc_500['suppressed'].sum()}  ({suppressed_pct:.1f}%)")
print(f"500 m — records shown          : {(~enc_500['suppressed']).sum()}")

500 m — unique geographic tiles : 249
500 m — tiles k=1              : 105  (42.2%)
500 m — tiles k>=5             : 15  (6.0%)
500 m — records suppressed     : 432  (83.7%)
500 m — records shown          : 84


In [7]:
# Figure 15d — Original positions at 500 m with k<5 suppression applied
# Colour by suppression status; both shown and suppressed use original coords
# so analysts can see exactly where records are hidden vs displayed.
enc_500_geo = enc_500.merge(df[["individual_id", "LAT", "LON"]], on="individual_id")

m15d = folium.Map(location=centre, zoom_start=13,
                  tiles="cartodbpositron", control_scale=True)

shown_fg      = folium.FeatureGroup(name="Shown (k ≥ 5)", show=True)
suppressed_fg = folium.FeatureGroup(name="Suppressed (k < 5)", show=True)

for _, row in enc_500_geo.iterrows():
    if row["suppressed"]:
        folium.CircleMarker(
            location=[row["LAT"], row["LON"]],
            radius=3, color="#888888", fill=True,
            fill_color="#888888", fill_opacity=0.4,
            popup=f"Suppressed — k={row['k']} (tile < {K_MIN} records)",
        ).add_to(suppressed_fg)
    else:
        folium.CircleMarker(
            location=[row["LAT"], row["LON"]],
            radius=3, color="#2171b5", fill=True,
            fill_color="#2171b5", fill_opacity=0.7,
            popup=f"k={row['k']}",
        ).add_to(shown_fg)

shown_fg.add_to(m15d)
suppressed_fg.add_to(m15d)

legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;background:white;'
    'padding:8px 12px;border-radius:6px;font-size:12px;'
    'border:1px solid #aaa;z-index:9999;">'
    f'<b>500 m + k≥{K_MIN} suppression</b><br>'
    f'<span style="color:#2171b5;">&#9679;</span> Shown on public map (k ≥ {K_MIN})<br>'
    f'<span style="color:#888888;">&#9679;</span> Suppressed from public map (k < {K_MIN})'
    '</div>'
)
m15d.get_root().html.add_child(folium.Element(legend_html))
plugins.Fullscreen(position="topleft").add_to(m15d)
folium.LayerControl(collapsed=False).add_to(m15d)
m15d

**Figure 15d.** Original incident locations coloured by 500 m suppression status.
Blue circles are records in tiles containing ≥ 5 individuals — these would appear
on the public-facing service planning map. Grey circles are suppressed records
(k < 5); they remain in the decode tier for licensed analysts but are withheld
from any public display. Toggle the suppressed layer off to see what a public
health analyst without decode access would observe. Compared to Figure 15b, the
grey area (suppressed) highlights where data sparsity — often in historically
under-served areas — creates gaps in the public map.

In [8]:
# Figure 15e — k-anonymity comparison: 250 m vs 500 m side by side
k_vals_500 = tile_counts_500["k"].clip(upper=10)
k_hist_500 = k_vals_500.value_counts().sort_index()

all_k = sorted(set(k_hist_250.index) | set(k_hist_500.index))
labels = [str(k) if k < 10 else "10+" for k in all_k]

fig_15e = go.Figure()
fig_15e.add_trace(go.Bar(
    name="250 m bins",
    x=labels,
    y=[k_hist_250.get(k, 0) for k in all_k],
    marker_color="#d94801", opacity=0.8,
))
fig_15e.add_trace(go.Bar(
    name="500 m bins",
    x=labels,
    y=[k_hist_500.get(k, 0) for k in all_k],
    marker_color="#2171b5", opacity=0.8,
))
fig_15e.update_layout(
    title=f"k-anonymity tile distribution: 250 m vs 500 m  (suppression threshold: k < {K_MIN})",
    xaxis_title="Records per tile (k)",
    yaxis_title="Number of tiles",
    barmode="group",
    template="plotly_white", height=400, width=650,
)
fig_15e.show()

**Figure 15e.** Side-by-side k-anonymity tile distributions for 250 m (orange)
and 500 m (blue) bin sizes. The dashed red line marks the k < 5 suppression
threshold. At 250 m, the majority of tiles are singletons; at 500 m, the
distribution shifts right — more tiles contain 5 or more records and the
singleton count drops substantially. Records in tiles left of the threshold
are suppressed from the public display map under the Scenario B rule.


---
## Part 4 — Comparison Summary


In [9]:
# Table 15a — Privacy–utility comparison: 250 m vs 500 m + suppression
shown_500 = (~enc_500["suppressed"]).sum()
suppressed_500 = enc_500["suppressed"].sum()

rows_tbl = [
    ["bin_size_m", "250 m", "500 m"],
    ["Unique encrypted tiles", str(n_tiles_250), str(n_tiles_500)],
    ["Tiles with k = 1 (singleton)", f"{(tile_counts_250['k']==1).sum()} ({k1_pct_250:.0f}% of tiles)", f"{(tile_counts_500['k']==1).sum()} ({k1_pct_500:.0f}% of tiles)"],
    ["Tiles with k ≥ 5", f"{(tile_counts_250['k']>=5).sum()} ({k5plus_pct_250:.0f}% of tiles)", f"{(tile_counts_500['k']>=5).sum()} ({k5plus_pct_500:.0f}% of tiles)"],
    ["Records shown on public map", "516 (100%)", f"{shown_500} ({shown_500/516*100:.0f}%)"],
    ["Records suppressed (k < 5)", "0 (0%)", f"{suppressed_500} ({suppressed_500/516*100:.0f}%)"],
    ["Re-ID risk (singleton records)", f"{(enc_250['k']==1).sum()} records fully exposed", f"{(enc_500[~enc_500['suppressed']]['k']==1).sum()} records in shown tiles with k=1"],
]

df_tbl = pd.DataFrame(rows_tbl[1:], columns=["Metric"] + rows_tbl[0][1:])
print("Table 15a — Privacy–utility comparison\n")
print(df_tbl.to_string(index=False))


Table 15a — Privacy–utility comparison

                        Metric                     250 m                             500 m
        Unique encrypted tiles                       416                               249
  Tiles with k = 1 (singleton)        330 (79% of tiles)                105 (42% of tiles)
              Tiles with k ≥ 5           0 (0% of tiles)                  15 (6% of tiles)
   Records shown on public map                516 (100%)                          84 (16%)
    Records suppressed (k < 5)                    0 (0%)                         432 (84%)
Re-ID risk (singleton records) 330 records fully exposed 0 records in shown tiles with k=1


---
## Part 5 — Ethical Framework for Scenario B

### Three Dominant Tensions (from NB10)

**1. Utility vs. disclosure risk**
Service planners need hotspot accuracy to site needle exchanges, naloxone
distribution points, and treatment referral hubs. But publishing individual
incident locations — even encrypted — creates a residual re-identification
risk for a population already subject to policing, housing discrimination, and
social stigma. The 500 m + suppression regime trades some spatial precision
for a meaningful reduction in singleton tiles.

**2. Equity vs. efficiency**
The PDPH 2022 report shows that overdose deaths increased 87 % among
non-Hispanic Black residents between 2018 and 2022. A suppression rule that
removes records from low-density tiles will disproportionately suppress deaths
in less dense areas — potentially making historically under-served
neighbourhoods invisible in the public map precisely because they have
fewer records. Suppression thresholds must be reviewed for differential
impact across demographic and geographic strata.

**3. Stewardship vs. capability**
The decode tier (prp\_key + aead\_key) provides public health analysts with
exact coordinates for epidemiological investigation. The NB10 recommendation
restricts this to *licensed analysts under a Data Use Agreement (DUA)*,
replicating the legal safeguards required for HIPAA-covered health data
under 45 CFR Part 164. Displaying tier access (jitter\_key only) is
appropriate for public-facing service planning maps.

### Implementation Checklist for Scenario B Deployments

| Control | Requirement |
|---------|------------|
| Bin size | ≥ 500 m; review if population density varies widely across the study area |
| Suppression | Suppress tiles with < 5 records from public map; log suppression counts for equity audit |
| Display tier | Jitter key only; distributable to community organisations |
| Decode tier | Full key pair; restricted to IRB-approved / DUA-covered analysts |
| Re-identification audit | Run k-anonymity analysis before each data release |
| Equity review | Check suppression rate by ZIP code and demographic group before publication |
| Retention | Encrypted records should be deleted after the operational period specified in the DUA |


---
## Glossary

| Term | Definition |
|------|------------|
| k-anonymity | Privacy model requiring each record to be indistinguishable from at least k − 1 others with respect to quasi-identifiers; here the quasi-identifier is encrypted tile membership |
| singleton tile | An encrypted tile containing exactly one record (k = 1); the sole occupant is fully re-identifiable by tile membership alone |
| suppression rule | Policy of withholding records in tiles below a minimum count threshold from the public display tier; reduces re-identification risk at the cost of spatial coverage |
| stigmatised population | A group whose social identity is discredited; substance use disorder is associated with policing, housing discrimination, and loss of employment, making spatial re-identification especially harmful |
| Data Use Agreement (DUA) | A legal contract governing the conditions under which sensitive data may be accessed, used, and shared; required for decode-tier access in Scenario B |
| HIPAA | Health Insurance Portability and Accountability Act (US, 1996); 45 CFR Part 164 sets the Privacy Rule governing protected health information |
| fentanyl | A synthetic opioid 50–100× more potent than morphine; detected in 96 % of opioid-involved overdose deaths in Philadelphia in 2022 |
| xylazine | A veterinary anesthetic increasingly found in street fentanyl supplies; detected in 34 % of all Philadelphia overdose deaths in 2022; not reversed by naloxone |
| harm reduction | Public health approach that reduces negative consequences of substance use without requiring abstinence; includes naloxone distribution, syringe services, and drug-checking |
| display tier | The encryption scheme tier that exposes only jittered positions (jitter\_key); safe for public-facing service planning maps |
| decode tier | The encryption scheme tier requiring both prp\_key and aead\_key; returns exact coordinates; restricted to licensed analysts |


---
## References

**Primary data sources**

Philadelphia Department of Public Health (2023). *Unintentional Drug Overdose
Fatalities in Philadelphia, 2022.* CHART Vol. 8, No. 3. September 2023.
https://www.phila.gov/media/20231002090544/CHARTv8e3.pdf

City of Philadelphia (2024). *Philadelphia ZIP Code Boundaries* [GeoJSON].
OpenDataPhilly / ArcGIS Hub.
https://opendataphilly.org/datasets/zip-codes/

**Background on Philadelphia's opioid crisis**

Friedman, J. R., & Hansen, H. (2022). Evaluation of increases in drug
overdose mortality rates in the US by race and ethnicity before and during
the COVID-19 pandemic. *JAMA Psychiatry, 79*(4), 379–381.
https://doi.org/10.1001/jamapsychiatry.2022.0004

Spencer, M. R., Miniño, A. M., & Garnett, M. F. (2023). Co-involvement of
opioids in drug overdose deaths involving cocaine and psychostimulants,
2011–2021. *NCHS Data Brief,* (474), 1–8.

**On k-anonymity and re-identification**

Sweeney, L. (2002). k-Anonymity: A model for protecting privacy.
*International Journal of Uncertainty, Fuzziness and Knowledge-Based
Systems, 10*(5), 557–570.

El Emam, K., Jonker, E., Arbuckle, L., & Malin, B. (2011). A systematic
review of re-identification attacks on health data. *PLOS ONE, 6*(12),
e28071. https://doi.org/10.1371/journal.pone.0028071

Matthews, G. J., & Harel, O. (2011). Data confidentiality: A review of
methods for statistical disclosure limitation and methods for assessing
privacy. *Statistics Surveys, 5,* 1–29.
https://doi.org/10.1214/11-SS074

**On ethics of geoprivacy for stigmatised populations**

Kearney, M. S., & Bhide, A. (2019). Measuring the effect of drug treatment
on geographic patterns of substance use disorder.
In *NBER Working Paper* 25940.

Genberg, B. L., et al. (2011). A comparison of HIV/AIDS-related
stigma in four countries: Negative attitudes and perceived acts of
discrimination towards people living with HIV/AIDS.
*Social Science & Medicine, 73*(11), 1600–1608.
https://doi.org/10.1016/j.socscimed.2011.09.018

**Regulatory and legal context**

U.S. Department of Health and Human Services (2024). *HIPAA for
Professionals.* 45 CFR Part 164 — Security and Privacy.
https://www.hhs.gov/hipaa/for-professionals/index.html

Substance Abuse and Mental Health Services Administration (2024).
*Confidentiality of Substance Use Disorder Patient Records, 42 CFR Part 2.*
https://www.samhsa.gov/about-us/who-we-are/laws-regulations/confidentiality-regulations-faqs
